# 02 — Feature Engineering
**Financial Risk Model | Derived Financial Indicators**

This notebook constructs domain-specific financial features that measurably
improve the model's ability to separate high-risk from low-risk borrowers.

Features engineered:
| Feature | Formula | Financial Meaning |
|---|---|---|
| DebtToIncome | Annual EMI / Income | Cash-flow pressure; > 0.43 = CFPB hard cap |
| LoanToIncome | Loan Amount / Income | Size of obligation vs earnings |
| EMIToIncome | Monthly EMI / Net Monthly Income | Monthly affordability |
| AssetToLiability | Total Assets / Total Liabilities | Solvency ratio |
| NetWorth | Assets − Liabilities | Absolute equity position |
| LoanBurdenRatio | Loan / (Income × Tenure) | Lifetime earnings consumed by loan |
| FinancialStressIndex | Weighted composite | Overall financial fragility score |
| CreditScoreBin | Binned credit score | Standard bureau band labels |
| AgeBin | Life-stage cohort | Behavioural risk segment |
| LoanAmountBin | Quantile-based tier | Loss severity proxy |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, os.path.join('..', 'utils'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_raw_data, generate_synthetic_dataset
from feature_engineering import engineer_features
from metrics import iv_woe

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Ready ✅')

## 1. Load Raw Data

In [ ]:
RAW_PATH = '../data/raw/financial_risk_data.csv'

if not os.path.exists(RAW_PATH):
    df_raw = generate_synthetic_dataset(n_samples=5000, save_path=RAW_PATH)
else:
    df_raw = load_raw_data(filepath=RAW_PATH, config_path='../config/config.yaml')

print(f'Raw: {df_raw.shape}')

## 2. Apply Feature Engineering Pipeline

In [ ]:
df = engineer_features(df_raw)

new_features = [c for c in df.columns if c not in df_raw.columns]
print(f'New features created ({len(new_features)}): {new_features}')
print(f'\nDataset shape after engineering: {df.shape}')

df[new_features].describe().T

## 3. Ratio Feature Distributions by Risk Class

In [ ]:
ratio_features = ['DebtToIncome', 'LoanToIncome', 'EMIToIncome',
                  'AssetToLiability', 'NetWorth', 'FinancialStressIndex']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors = {0: '#4CAF50', 1: '#F44336'}
labels = {0: 'Low Risk', 1: 'High Risk'}

for i, feat in enumerate(ratio_features):
    ax = axes[i]
    for flag in [0, 1]:
        subset = df[df['RiskFlag'] == flag][feat].dropna()
        # Clip extreme outliers for display only
        subset = subset.clip(subset.quantile(0.01), subset.quantile(0.99))
        subset.plot.kde(ax=ax, color=colors[flag], label=labels[flag], linewidth=2.5, alpha=0.85)
        ax.axvline(subset.median(), color=colors[flag], linestyle='--', linewidth=1.2, alpha=0.6)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlabel('')

plt.suptitle('Engineered Ratio Features — Distribution by Risk Class',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/engineered_features_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Debt-to-Income Threshold Analysis
The CFPB Qualified Mortgage rule caps DTI at 43%. Explore the default rate around this threshold.

In [ ]:
dti_bins = pd.cut(df['DebtToIncome'].clip(0, 2),
                  bins=[0, 0.2, 0.36, 0.43, 0.5, 0.65, 2.0],
                  labels=['0-20%', '20-36%', '36-43%', '43-50%', '50-65%', '>65%'])

dti_stats = df.groupby(dti_bins)['RiskFlag'].agg(['mean', 'count']).reset_index()
dti_stats.columns = ['DTI Band', 'Default Rate', 'Count']
dti_stats['Default Rate %'] = (dti_stats['Default Rate'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bars = ax1.bar(dti_stats['DTI Band'], dti_stats['Default Rate %'],
               color=plt.cm.RdYlGn_r(dti_stats['Default Rate'].values),
               alpha=0.85, edgecolor='white')
ax2.plot(dti_stats['DTI Band'], dti_stats['Count'], 'o--',
         color='#2196F3', linewidth=2, markersize=7, label='Count')

ax1.axvline(x=2.5, color='black', linestyle=':', linewidth=2, label='CFPB 43% cap')
ax1.set_xlabel('Debt-to-Income Band', fontsize=12)
ax1.set_ylabel('Default Rate (%)', fontsize=12)
ax2.set_ylabel('Number of Customers', fontsize=12, color='#2196F3')
ax1.set_title('Default Rate vs Debt-to-Income Band', fontsize=13, fontweight='bold')

for bar, val in zip(bars, dti_stats['Default Rate %']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/dti_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(dti_stats.to_string(index=False))

## 5. Credit Score Band Analysis

In [ ]:
cs_stats = df.groupby('CreditScoreBin')['RiskFlag'].agg(['mean', 'count']).reset_index()
cs_stats.columns = ['Credit Band', 'Default Rate', 'Count']

band_order = ['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent']
cs_stats = cs_stats.set_index('Credit Band').reindex(band_order).reset_index()
cs_stats['Default Rate %'] = (cs_stats['Default Rate'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
palette = ['#D32F2F', '#FF7043', '#FFC107', '#66BB6A', '#1565C0']
bars = ax.bar(cs_stats['Credit Band'], cs_stats['Default Rate %'],
              color=palette, edgecolor='white', linewidth=1.5)

for bar, val, cnt in zip(bars, cs_stats['Default Rate %'], cs_stats['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val}%\nn={int(cnt):,}', ha='center', fontsize=9)

ax.set_xlabel('Credit Score Band', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.set_title('Default Rate by Credit Score Band', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/credit_score_bands.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Information Value (IV) — Feature Predictive Power

In [ ]:
iv_results = {}
numeric_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'ExistingLoansCount',
                     'DebtToIncome', 'LoanToIncome', 'EMIToIncome',
                     'AssetToLiability', 'NetWorth', 'FinancialStressIndex']

for feat in numeric_features:
    try:
        iv_val, _ = iv_woe(df, feat, target_col='RiskFlag', bins=10)
        iv_results[feat] = iv_val
    except Exception as e:
        iv_results[feat] = 0.0

iv_df = pd.DataFrame.from_dict(iv_results, orient='index', columns=['IV'])
iv_df = iv_df.sort_values('IV', ascending=False)

def iv_label(iv):
    if iv < 0.02:  return 'Unpredictive'
    elif iv < 0.1: return 'Weak'
    elif iv < 0.3: return 'Medium'
    elif iv < 0.5: return 'Strong'
    else:          return 'Very Strong / Check for leakage'

iv_df['Strength'] = iv_df['IV'].apply(iv_label)
print(iv_df.to_string())

fig, ax = plt.subplots(figsize=(10, 7))
colors_iv = ['#D32F2F' if v > 0.3 else '#FF9800' if v > 0.1 else '#4CAF50'
             for v in iv_df['IV']]
ax.barh(iv_df.index, iv_df['IV'], color=colors_iv, edgecolor='white')
ax.axvline(0.02, color='grey', linestyle='--', linewidth=1, label='IV=0.02 (weak threshold)')
ax.axvline(0.1,  color='orange', linestyle='--', linewidth=1, label='IV=0.10 (medium threshold)')
ax.axvline(0.3,  color='red', linestyle='--', linewidth=1, label='IV=0.30 (strong threshold)')
ax.set_xlabel('Information Value (IV)', fontsize=12)
ax.set_title('Feature Predictive Power — Information Value', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/information_value.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Financial Stress Index vs Default Rate

In [ ]:
df['FSI_decile'] = pd.qcut(df['FinancialStressIndex'], q=10, labels=False, duplicates='drop') + 1

fsi_stats = df.groupby('FSI_decile')['RiskFlag'].agg(['mean', 'count']).reset_index()
fsi_stats.columns = ['FSI Decile', 'Default Rate', 'Count']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fsi_stats['FSI Decile'], fsi_stats['Default Rate'] * 100,
        'o-', color='#F44336', linewidth=2.5, markersize=8)
ax.fill_between(fsi_stats['FSI Decile'], fsi_stats['Default Rate'] * 100,
                alpha=0.15, color='#F44336')
ax.set_xlabel('Financial Stress Index Decile (1=Lowest Stress, 10=Highest)', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.set_title('Default Rate vs Financial Stress Index Decile', fontsize=13, fontweight='bold')
ax.set_xticks(range(1, 11))
plt.tight_layout()
plt.savefig('../reports/figures/fsi_default_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Processed Dataset

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/processed_data.csv', index=False)
print(f'✅ Processed dataset saved: {df.shape}')
print(f'   Columns: {list(df.columns)}')
print('\nNext step → Model Training (notebook 03)')